In [ ]:
###Basic packages
import pandas as pd
import numpy as np

In [57]:
###Relevant UQ packages
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
import torch.nn as nn
from pyro.infer import Predictive

###PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

###Forward and Backward Selection
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
import statsmodels.api as sm

# HMC
from pyro.infer import MCMC, NUTS

# variational inference
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from tqdm.auto import trange

import matplotlib as mpl
import os
import sys
import math

##### DATA CLEANING

#### ASML

In [29]:
df = pd.read_csv('denoised_ASML.csv')
df = df[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
df.dropna(inplace=True)

df.head()
df.count()

datadate    1214
cshoc       1214
prcld       1214
prcstd      1214
qunit       1214
trfd        1214
prccd       1214
dtype: int64

In [30]:
rows, cols = df.shape
print(f"Rows: {rows}, Columns: {cols}")

Rows: 1214, Columns: 7


In [31]:
###Split the data into pre and post GDPR in 2016 and 2018

###Pre-GDPR data April 2016

df_pre = df[df['datadate'] < '2016-05-02'].reset_index(drop=True)

###Post-GDPR data April 2018

df_post = df[df['datadate'] > '2018-06-01'].reset_index(drop=True)


In [32]:
df_pre, df_post

(       datadate        cshoc  prcld  prcstd  qunit      trfd      prccd
 0    2015-04-02  438271869.0  92.74    10.0    1.0  1.501852  93.557559
 1    2015-04-07  438271869.0  91.65    10.0    1.0  1.501852  91.859711
 2    2015-04-08  438271869.0  91.72    10.0    1.0  1.501852  90.432725
 3    2015-04-09  438271869.0  92.67    10.0    1.0  1.501852  90.762329
 4    2015-04-10  438271869.0  93.54    10.0    1.0  1.501852  90.791163
 ..          ...          ...    ...     ...    ...       ...        ...
 270  2016-04-25  433332427.0  85.07    10.0    1.0  1.512459  80.873233
 271  2016-04-26  433332427.0  84.78    10.0    1.0  1.512459  80.818181
 272  2016-04-27  433332427.0  85.23    10.0    1.0  1.512459  81.832749
 273  2016-04-28  433332427.0  85.71    10.0    1.0  1.512459  82.382968
 274  2016-04-29  433332427.0  84.23    10.0    1.0  1.512459  81.477715
 
 [275 rows x 7 columns],
        datadate        cshoc   prcld  prcstd  qunit      trfd       prccd
 0    2018-06-04  4314

In [33]:
feature_list_pre = []

###Drop date
df_pre.drop(columns=['datadate'], inplace=True)

for i in range(5, len(df) - 1):
    five_days_chunk = df_pre.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre.append(chunk_features_flatten)

In [34]:
target_list_pre = []

for i in range(5, len(df_pre) - 1):
    target = df_pre.iloc[i]["prccd"]
    target_list_pre.append(target)

In [35]:
feature_list_pre, target_list_pre

([array([4.38271869e+08, 9.27400000e+01, 1.00000000e+01, 1.00000000e+00,
         1.50185161e+00, 4.38271869e+08, 9.16500000e+01, 1.00000000e+01,
         1.00000000e+00, 1.50185161e+00, 4.38271869e+08, 9.17200000e+01,
         1.00000000e+01, 1.00000000e+00, 1.50185161e+00, 4.38271869e+08,
         9.26700000e+01, 1.00000000e+01, 1.00000000e+00, 1.50185161e+00,
         4.38271869e+08, 9.35400000e+01, 1.00000000e+01, 1.00000000e+00,
         1.50185161e+00]),
  array([4.38271869e+08, 9.16500000e+01, 1.00000000e+01, 1.00000000e+00,
         1.50185161e+00, 4.38271869e+08, 9.17200000e+01, 1.00000000e+01,
         1.00000000e+00, 1.50185161e+00, 4.38271869e+08, 9.26700000e+01,
         1.00000000e+01, 1.00000000e+00, 1.50185161e+00, 4.38271869e+08,
         9.35400000e+01, 1.00000000e+01, 1.00000000e+00, 1.50185161e+00,
         4.38271869e+08, 9.35800000e+01, 1.00000000e+01, 1.00000000e+00,
         1.50185161e+00]),
  array([4.38271869e+08, 9.17200000e+01, 1.00000000e+01, 1.00000000e+0

In [36]:
df_features_pre = pd.DataFrame(feature_list_pre)

# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre = pd.DataFrame(target_list_pre, columns=['target'])

##Dataframe
df_pre_final = pd.concat([df_features_pre, df_targets_pre], axis=1)
df_pre_final.to_csv("pregdprApril2016.csv", index=False)


In [37]:
feature_list_post = []

###Drop date
df_post.drop(columns=['datadate'], inplace=True)

for i in range(5, len(df) - 1):
    five_days_chunk = df_post.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post.append(chunk_features_flatten)

In [38]:
target_list_post = []

for i in range(5, len(df_pre) - 1):
    target = df_post.iloc[i]["prccd"]
    target_list_post.append(target)

In [40]:
df_features_post = pd.DataFrame(feature_list_post)

# Convert target_list_pre to a DataFrame (with one column)
df_targets_post = pd.DataFrame(target_list_post, columns=['target'])

##Dataframe
df_post_final = pd.concat([df_features_post, df_targets_post], axis=1)
df_post_final.to_csv("postgdprMay2018.csv", index=False)

#### ATOS

In [41]:
###Reading datasets
df_atos = pd.read_csv('denoised_ATOS.csv')
df_atos = df_atos[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
df_atos.dropna(inplace=True)
df_atos.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,101332527,64.06,10,1,1.059301,65.052770
4,2015-04-07,101332527,64.69,10,1,1.059301,64.681143
5,2015-04-08,101332527,64.55,10,1,1.059301,64.337083
6,2015-04-09,101332527,64.90,10,1,1.059301,65.366597
7,2015-04-10,101332527,66.81,10,1,1.059301,65.274990


In [54]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_atos = df_atos[df_atos['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_atos = df_atos[df_atos['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_atos, df_post_atos

(       datadate      cshoc  prcld  prcstd  qunit      trfd      prccd
 0    2015-04-02  101332527  64.06      10      1  1.059301  65.052770
 1    2015-04-07  101332527  64.69      10      1  1.059301  64.681143
 2    2015-04-08  101332527  64.55      10      1  1.059301  64.337083
 3    2015-04-09  101332527  64.90      10      1  1.059301  65.366597
 4    2015-04-10  101332527  66.81      10      1  1.059301  65.274990
 ..          ...        ...    ...     ...    ...       ...        ...
 270  2016-04-25  103523793  78.01      10      1  1.071445  89.071675
 271  2016-04-26  103523793  77.65      10      1  1.071445  88.724621
 272  2016-04-27  103523793  77.72      10      1  1.071445  89.029254
 273  2016-04-28  103691591  77.27      10      1  1.071445  89.096223
 274  2016-04-29  103691591  77.03      10      1  1.071445  90.307504
 
 [275 rows x 7 columns],
        datadate      cshoc   prcld  prcstd  qunit      trfd       prccd
 0    2018-06-04  105598479  117.15      10     

In [55]:
###Feature List
feature_list_pre_atos = []
###Drop date
df_pre_atos.drop(columns=['datadate'], inplace=True)
for i in range(5, len(df_atos) - 1):
    five_days_chunk = df_pre_atos.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_atos.append(chunk_features_flatten)

###Target List
target_list_pre_atos = []
for i in range(5, len(df_pre_atos) - 1):
    target = df_pre_atos.iloc[i]["prccd"]
    target_list_pre_atos.append(target)

In [56]:
feature_list_pre_atos, target_list_pre_atos

([array([1.01332527e+08, 6.40600000e+01, 1.00000000e+01, 1.00000000e+00,
         1.05930099e+00, 1.01332527e+08, 6.46900000e+01, 1.00000000e+01,
         1.00000000e+00, 1.05930099e+00, 1.01332527e+08, 6.45500000e+01,
         1.00000000e+01, 1.00000000e+00, 1.05930099e+00, 1.01332527e+08,
         6.49000000e+01, 1.00000000e+01, 1.00000000e+00, 1.05930099e+00,
         1.01332527e+08, 6.68100000e+01, 1.00000000e+01, 1.00000000e+00,
         1.05930099e+00]),
  array([1.01332527e+08, 6.46900000e+01, 1.00000000e+01, 1.00000000e+00,
         1.05930099e+00, 1.01332527e+08, 6.45500000e+01, 1.00000000e+01,
         1.00000000e+00, 1.05930099e+00, 1.01332527e+08, 6.49000000e+01,
         1.00000000e+01, 1.00000000e+00, 1.05930099e+00, 1.01332527e+08,
         6.68100000e+01, 1.00000000e+01, 1.00000000e+00, 1.05930099e+00,
         1.01332527e+08, 6.76200000e+01, 1.00000000e+01, 1.00000000e+00,
         1.05930099e+00]),
  array([1.01332527e+08, 6.45500000e+01, 1.00000000e+01, 1.00000000e+0

In [58]:
df_features_pre_atos = pd.DataFrame(feature_list_pre_atos)

# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_atos = pd.DataFrame(target_list_pre_atos, columns=['target'])

##Dataframe
df_pre_final_atos = pd.concat([df_features_pre_atos, df_targets_pre_atos], axis=1)
df_pre_final_atos.to_csv("pregdprApril2016_ATOS.csv", index=False)

In [59]:
feature_list_post_atos = []
###Drop date
df_post_atos.drop(columns=['datadate'], inplace=True)
for i in range(5, len(df_atos) - 1):
    five_days_chunk = df_post_atos.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_atos.append(chunk_features_flatten)
###Target List
target_list_post_atos = []
for i in range(5, len(df_post_atos) - 1):
    target = df_post_atos.iloc[i]["prccd"]
    target_list_post_atos.append(target)
df_features_post_atos = pd.DataFrame(feature_list_post_atos)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_atos = pd.DataFrame(target_list_post_atos, columns=['target'])
##Dataframe
df_post_final_atos = pd.concat([df_features_post_atos, df_targets_post_atos], axis=1)
df_post_final_atos.to_csv("postgdprMay2018_ATOS.csv", index=False)

#### Dassault

In [49]:
df_dassault = pd.read_csv('denoised_Dassault.csv')
df_dassault = df_dassault[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
df_dassault.dropna(inplace=True)
df_dassault.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,255723498.0,61.03,10.0,1.0,1.163192,61.786534
4,2015-04-07,255723498.0,61.73,10.0,1.0,1.163192,61.920245
5,2015-04-08,255230678.0,62.01,10.0,1.0,1.163192,61.018482
6,2015-04-09,255230678.0,61.83,10.0,1.0,1.163192,61.834978
7,2015-04-10,255230678.0,63.35,10.0,1.0,1.163192,61.523115


In [60]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_dassault = df_dassault[df_dassault['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_dassault = df_dassault[df_dassault['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_dassault, df_post_dassault
###Feature List
feature_list_pre_dassault = []
###Drop date
df_pre_dassault.drop(columns=['datadate'], inplace=True)
for i in range(5, len(df_dassault) - 1):
    five_days_chunk = df_pre_dassault.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_dassault.append(chunk_features_flatten)
###Target List
target_list_pre_dassault = []
for i in range(5, len(df_pre_dassault) - 1):
    target = df_pre_dassault.iloc[i]["prccd"]
    target_list_pre_dassault.append(target)
feature_list_pre_dassault, target_list_pre_dassault
df_features_pre_dassault = pd.DataFrame(feature_list_pre_dassault)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_dassault = pd.DataFrame(target_list_pre_dassault, columns=['target'])
##Dataframe
df_pre_final_dassault = pd.concat([df_features_pre_dassault, df_targets_pre_dassault], axis=1)
df_pre_final_dassault.to_csv("pregdprApril2016_Dassault.csv", index=False)
feature_list_post_dassault = []
###Drop date
df_post_dassault.drop(columns=['datadate'], inplace=True)
for i in range(5, len(df_dassault) - 1):
    five_days_chunk = df_post_dassault.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_dassault.append(chunk_features_flatten)
###Target List
target_list_post_dassault = []
for i in range(5, len(df_post_dassault) - 1):
    target = df_post_dassault.iloc[i]["prccd"]
    target_list_post_dassault.append(target)
df_features_post_dassault = pd.DataFrame(feature_list_post_dassault)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_dassault = pd.DataFrame(target_list_post_dassault, columns=['target'])
##Dataframe
df_post_final_dassault = pd.concat([df_features_post_dassault, df_targets_post_dassault], axis=1)
df_post_final_dassault.to_csv("postgdprMay2018_Dassault.csv", index=False)

#### Ericsson

In [43]:
def_ericsson = pd.read_csv('denoised_Ericcson.csv')
def_ericsson = def_ericsson[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
def_ericsson.dropna(inplace=True)
def_ericsson.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,261755983.0,101.0,10.0,1.0,1.535461,102.505721
4,2015-04-07,261755983.0,102.8,10.0,1.0,1.535461,103.508487
5,2015-04-08,261755983.0,103.0,10.0,1.0,1.535461,102.383602
6,2015-04-09,261755983.0,103.8,10.0,1.0,1.535461,103.538812
7,2015-04-10,261755983.0,104.8,10.0,1.0,1.535461,103.464017


In [61]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_ericsson = def_ericsson[def_ericsson['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_ericsson = def_ericsson[def_ericsson['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_ericsson, df_post_ericsson
###Feature List
feature_list_pre_ericsson = []
###Drop date
df_pre_ericsson.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_ericsson) - 1):
    five_days_chunk = df_pre_ericsson.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_ericsson.append(chunk_features_flatten)
###Target List
target_list_pre_ericsson = []
for i in range(5, len(df_pre_ericsson) - 1):
    target = df_pre_ericsson.iloc[i]["prccd"]
    target_list_pre_ericsson.append(target)
feature_list_pre_ericsson, target_list_pre_ericsson
df_features_pre_ericsson = pd.DataFrame(feature_list_pre_ericsson)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_ericsson = pd.DataFrame(target_list_pre_ericsson, columns=['target'])
##Dataframe
df_pre_final_ericsson = pd.concat([df_features_pre_ericsson, df_targets_pre_ericsson], axis=1)
df_pre_final_ericsson.to_csv("pregdprApril2016_Ericcson.csv", index=False)
feature_list_post_ericsson = []
###Drop date
df_post_ericsson.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_ericsson) - 1):
    five_days_chunk = df_post_ericsson.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_ericsson.append(chunk_features_flatten)
###Target List
target_list_post_ericsson = []
for i in range(5, len(df_post_ericsson) - 1):
    target = df_post_ericsson.iloc[i]["prccd"]
    target_list_post_ericsson.append(target)
df_features_post_ericsson = pd.DataFrame(feature_list_post_ericsson)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_ericsson = pd.DataFrame(target_list_post_ericsson, columns=['target'])
##Dataframe
df_post_final_ericsson = pd.concat([df_features_post_ericsson, df_targets_post_ericsson], axis=1)
df_post_final_ericsson.to_csv("postgdprMay2018_Ericcson.csv", index=False)

#### Hexagon

In [44]:
def_hexagon = pd.read_csv('denoised_HexagonAB.csv')
def_hexagon = def_hexagon[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
def_hexagon.dropna(inplace=True)
def_hexagon.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,163592949,76.57,10,1,1.377609,77.295300
4,2015-04-07,163592949,76.74,10,1,1.377609,77.443517
5,2015-04-08,163592949,77.01,10,1,1.377609,76.701238
6,2015-04-09,163592949,77.98,10,1,1.377609,77.082956
7,2015-04-10,163592949,79.13,10,1,1.377609,76.565748


In [62]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_hexagon = def_hexagon[def_hexagon['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_hexagon = def_hexagon[def_hexagon['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_hexagon, df_post_hexagon
###Feature List
feature_list_pre_hexagon = []
###Drop date
df_pre_hexagon.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_hexagon) - 1):
    five_days_chunk = df_pre_hexagon.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_hexagon.append(chunk_features_flatten)
###Target List
target_list_pre_hexagon = []
for i in range(5, len(df_pre_hexagon) - 1):
    target = df_pre_hexagon.iloc[i]["prccd"]
    target_list_pre_hexagon.append(target)
feature_list_pre_hexagon, target_list_pre_hexagon
df_features_pre_hexagon = pd.DataFrame(feature_list_pre_hexagon)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_hexagon = pd.DataFrame(target_list_pre_hexagon, columns=['target'])
##Dataframe
df_pre_final_hexagon = pd.concat([df_features_pre_hexagon, df_targets_pre_hexagon], axis=1)
df_pre_final_hexagon.to_csv("pregdprApril2016_Hexagon.csv", index=False)
feature_list_post_hexagon = []
###Drop date
df_post_hexagon.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_hexagon) - 1):
    five_days_chunk = df_post_hexagon.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_hexagon.append(chunk_features_flatten)
###Target List
target_list_post_hexagon = []
for i in range(5, len(df_post_hexagon) - 1):
    target = df_post_hexagon.iloc[i]["prccd"]
    target_list_post_hexagon.append(target)
df_features_post_hexagon = pd.DataFrame(feature_list_post_hexagon)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_hexagon = pd.DataFrame(target_list_post_hexagon, columns=['target'])
##Dataframe
df_post_final_hexagon = pd.concat([df_features_post_hexagon, df_targets_post_hexagon], axis=1)
df_post_final_hexagon.to_csv("postgdprMay2018_Hexagon.csv", index=False)

#### Infineon

In [45]:
def_infineon = pd.read_csv('denoised_Infineon.csv')
def_infineon = def_infineon[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
def_infineon.dropna(inplace=True)
def_infineon.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,1.128878e+09,11.125,10,1,1.099981,11.330205
4,2015-04-07,1.128878e+09,11.295,10,1,1.099981,11.358241
5,2015-04-08,1.128878e+09,11.300,10,1,1.099981,11.265135
6,2015-04-09,1.128878e+09,11.310,10,1,1.099981,11.241022
7,2015-04-10,1.128878e+09,11.585,10,1,1.099981,11.364222


In [63]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_infineon = def_infineon[def_infineon['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_infineon = def_infineon[def_infineon['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_infineon, df_post_infineon
###Feature List
feature_list_pre_infineon = []
###Drop date
df_pre_infineon.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_infineon) - 1):
    five_days_chunk = df_pre_infineon.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_infineon.append(chunk_features_flatten)
###Target List
target_list_pre_infineon = []
for i in range(5, len(df_pre_infineon) - 1):
    target = df_pre_infineon.iloc[i]["prccd"]
    target_list_pre_infineon.append(target)
feature_list_pre_infineon, target_list_pre_infineon
df_features_pre_infineon = pd.DataFrame(feature_list_pre_infineon)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_infineon = pd.DataFrame(target_list_pre_infineon, columns=['target'])
##Dataframe
df_pre_final_infineon = pd.concat([df_features_pre_infineon, df_targets_pre_infineon], axis=1)
df_pre_final_infineon.to_csv("pregdprApril2016_Infineon.csv", index=False)
feature_list_post_infineon = []
###Drop date
df_post_infineon.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_infineon) - 1):
    five_days_chunk = df_post_infineon.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_infineon.append(chunk_features_flatten)
###Target List
target_list_post_infineon = []
for i in range(5, len(df_post_infineon) - 1):
    target = df_post_infineon.iloc[i]["prccd"]
    target_list_post_infineon.append(target)
df_features_post_infineon = pd.DataFrame(feature_list_post_infineon)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_infineon = pd.DataFrame(target_list_post_infineon, columns=['target'])
##Dataframe
df_post_final_infineon = pd.concat([df_features_post_infineon, df_targets_post_infineon], axis=1)
df_post_final_infineon.to_csv("postgdprMay2018_Infineon.csv", index=False)

#### SAP

In [46]:
def_sap = pd.read_csv('denoised_SAP.csv')
def_sap = def_sap[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
def_sap.dropna(inplace=True)
def_sap.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,1.228504e+09,66.71,10,1,1.612828,66.922970
4,2015-04-07,1.228504e+09,66.85,10,1,1.612828,66.343573
5,2015-04-08,1.228504e+09,67.35,10,1,1.612828,65.930234
6,2015-04-09,1.228504e+09,67.45,10,1,1.612828,65.897010
7,2015-04-10,1.228504e+09,68.55,10,1,1.612828,66.159248


In [64]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_sap = def_sap[def_sap['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_sap = def_sap[def_sap['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_sap, df_post_sap
###Feature List
feature_list_pre_sap = []
###Drop date
df_pre_sap.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_sap) - 1):
    five_days_chunk = df_pre_sap.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_sap.append(chunk_features_flatten)
###Target List
target_list_pre_sap = []
for i in range(5, len(df_pre_sap) - 1):
    target = df_pre_sap.iloc[i]["prccd"]
    target_list_pre_sap.append(target)
feature_list_pre_sap, target_list_pre_sap
df_features_pre_sap = pd.DataFrame(feature_list_pre_sap)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_sap = pd.DataFrame(target_list_pre_sap, columns=['target'])
##Dataframe
df_pre_final_sap = pd.concat([df_features_pre_sap, df_targets_pre_sap], axis=1)
df_pre_final_sap.to_csv("pregdprApril2016_SAP.csv", index=False)
feature_list_post_sap = []
###Drop date
df_post_sap.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_sap) - 1):
    five_days_chunk = df_post_sap.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_sap.append(chunk_features_flatten)
###Target List
target_list_post_sap = []
for i in range(5, len(df_post_sap) - 1):
    target = df_post_sap.iloc[i]["prccd"]
    target_list_post_sap.append(target)
df_features_post_sap = pd.DataFrame(feature_list_post_sap)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_sap = pd.DataFrame(target_list_post_sap, columns=['target'])
##Dataframe
df_post_final_sap = pd.concat([df_features_post_sap, df_targets_post_sap], axis=1)
df_post_final_sap.to_csv("postgdprMay2018_SAP.csv", index=False)

#### STM

In [47]:
def_stm = pd.read_csv('denoised_STM.csv')
def_stm = def_stm[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
def_stm.dropna(inplace=True)
def_stm.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,910559805.0,8.516,10.0,1.0,1.421215,8.678764
4,2015-04-07,910559805.0,8.569,10.0,1.0,1.421215,8.470681
5,2015-04-08,910559805.0,8.398,10.0,1.0,1.421215,8.344882
6,2015-04-09,910559805.0,8.520,10.0,1.0,1.421215,8.353769
7,2015-04-10,910559805.0,8.667,10.0,1.0,1.421215,8.477795


In [65]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_stm = def_stm[def_stm['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_stm = def_stm[def_stm['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_stm, df_post_stm
###Feature List
feature_list_pre_stm = []
###Drop date
df_pre_stm.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_stm) - 1):
    five_days_chunk = df_pre_stm.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_stm.append(chunk_features_flatten)
###Target List
target_list_pre_stm = []
for i in range(5, len(df_pre_stm) - 1):
    target = df_pre_stm.iloc[i]["prccd"]
    target_list_pre_stm.append(target)
feature_list_pre_stm, target_list_pre_stm
df_features_pre_stm = pd.DataFrame(feature_list_pre_stm)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_stm = pd.DataFrame(target_list_pre_stm, columns=['target'])
##Dataframe
df_pre_final_stm = pd.concat([df_features_pre_stm, df_targets_pre_stm], axis=1)
df_pre_final_stm.to_csv("pregdprApril2016_STM.csv", index=False)
feature_list_post_stm = []
###Drop date
df_post_stm.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_stm) - 1):
    five_days_chunk = df_post_stm.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_stm.append(chunk_features_flatten)
###Target List
target_list_post_stm = []
for i in range(5, len(df_post_stm) - 1):
    target = df_post_stm.iloc[i]["prccd"]
    target_list_post_stm.append(target)
df_features_post_stm = pd.DataFrame(feature_list_post_stm)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_stm = pd.DataFrame(target_list_post_stm, columns=['target'])
##Dataframe
df_post_final_stm = pd.concat([df_features_post_stm, df_targets_post_stm], axis=1)
df_post_final_stm.to_csv("postgdprMay2018_STM.csv", index=False)

#### Vodafone

In [48]:
def_vodafone = pd.read_csv('denoised_Vodafone.csv')
def_vodafone = def_vodafone[['datadate', 'cshoc', 'prcld', 'prcstd', 'qunit', 'trfd', 'prccd']]
def_vodafone.dropna(inplace=True)
def_vodafone.head()

,datadate,cshoc,prcld,prcstd,qunit,trfd,prccd
1,2015-04-02,2.651204e+10,2.1980,10,1,4.333936,2.249970
4,2015-04-07,2.651204e+10,2.2114,10,1,4.333936,2.255409
5,2015-04-08,2.651204e+10,2.2240,10,1,4.333936,2.226589
6,2015-04-09,2.651204e+10,2.2200,10,1,4.333936,2.239524
7,2015-04-10,2.651204e+10,2.2530,10,1,4.333936,2.259484


In [66]:
####Spilt the data into pre and post GDPR in 2016 and 2018
###Pre-GDPR data April 2016
df_pre_vodafone = def_vodafone[def_vodafone['datadate'] < '2016-05-02'].reset_index(drop=True)
###Post-GDPR data April 2018
df_post_vodafone = def_vodafone[def_vodafone['datadate'] > '2018-06-01'].reset_index(drop=True)
df_pre_vodafone, df_post_vodafone
###Feature List
feature_list_pre_vodafone = []
###Drop date
df_pre_vodafone.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_vodafone) - 1):
    five_days_chunk = df_pre_vodafone.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_pre_vodafone.append(chunk_features_flatten)
###Target List
target_list_pre_vodafone = []
for i in range(5, len(df_pre_vodafone) - 1):
    target = df_pre_vodafone.iloc[i]["prccd"]
    target_list_pre_vodafone.append(target)
feature_list_pre_vodafone, target_list_pre_vodafone
df_features_pre_vodafone = pd.DataFrame(feature_list_pre_vodafone)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_pre_vodafone = pd.DataFrame(target_list_pre_vodafone, columns=['target'])
##Dataframe
df_pre_final_vodafone = pd.concat([df_features_pre_vodafone, df_targets_pre_vodafone], axis=1)
df_pre_final_vodafone.to_csv("pregdprApril2016_Vodafone.csv", index=False)
feature_list_post_vodafone = []
###Drop date
df_post_vodafone.drop(columns=['datadate'], inplace=True)
for i in range(5, len(def_vodafone) - 1):
    five_days_chunk = df_post_vodafone.iloc[i - 5 : i]
    chunk_features = five_days_chunk[['cshoc', 'prcld', 'prcstd', 'qunit', 'trfd']].values
    chunk_features_flatten = chunk_features.flatten()
    feature_list_post_vodafone.append(chunk_features_flatten)
###Target List
target_list_post_vodafone = []
for i in range(5, len(df_post_vodafone) - 1):
    target = df_post_vodafone.iloc[i]["prccd"]
    target_list_post_vodafone.append(target)
df_features_post_vodafone = pd.DataFrame(feature_list_post_vodafone)
# Convert target_list_pre to a DataFrame (with one column)
df_targets_post_vodafone = pd.DataFrame(target_list_post_vodafone, columns=['target'])
##Dataframe
df_post_final_vodafone = pd.concat([df_features_post_vodafone, df_targets_post_vodafone], axis=1)
df_post_final_vodafone.to_csv("postgdprMay2018_Vodafone.csv", index=False)